# LatentDriver Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_colab_runner.ipynb)

Use this notebook as a terminal-style launcher. Notebook cells are shell commands; clone/pull, Drive binding, and execution logic live in `scripts/*.py` and `src/`.


## Operating Model

1. Run the Drive mount code cell. This is a Colab platform exception, not experiment logic.
2. Run the GCS auth code cell if you want to evaluate directly from `gs://waymo_open_dataset_motion_v_1_1_0`. Skip it only if you intentionally use the Drive-local `assets/raw_womd` cache.
3. Run the bootstrap shell cell.
4. Optionally list profiles or check preprocessing markers.
5. Run the full reactive single-model gate cell. This is the next milestone after successful full preprocessing and dry-run readiness.
6. Inspect the debug status shell cell if a profile fails.

Only the first two code cells are notebook-native Python. Every other cell stays shell-first so the runnable logic remains in `scripts/` and `src/`.


## Mount Google Drive

Mount Drive once per Colab runtime so checkpoints, preprocessed artifacts, run bundles, and debug outputs can bind to the persistent project folder.


In [ ]:
from pathlib import Path

DRIVE_MOUNTPOINT = "/content/drive"
drive_root = Path(DRIVE_MOUNTPOINT) / "MyDrive"

if drive_root.is_dir():
    print(f"Drive already mounted at {drive_root}")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNTPOINT)
    print(f"Mounted Drive at {drive_root}")


## Authenticate GCS Access

Authenticate the Colab runtime for direct `gs://waymo_open_dataset_motion_v_1_1_0` reads. Keep this enabled unless you intentionally switch to the Drive-local raw WOMD cache.


In [ ]:
from __future__ import annotations

import json
import subprocess

USE_GCS_DIRECT = True

if USE_GCS_DIRECT:
    from google.colab import auth
    auth.authenticate_user()
    checks = {}
    commands = {
        "active_account": ["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=json"],
        "adc_token": ["gcloud", "auth", "application-default", "print-access-token"],
    }
    for name, command in commands.items():
        proc = subprocess.run(command, text=True, capture_output=True, check=False)
        payload = {"command": command, "returncode": proc.returncode}
        if name == "adc_token":
            payload["token_ready"] = proc.returncode == 0 and bool(proc.stdout.strip())
            payload["stderr"] = proc.stderr.strip()
        else:
            payload["stdout"] = proc.stdout.strip()
            payload["stderr"] = proc.stderr.strip()
        checks[name] = payload
    print(json.dumps({"use_gcs_direct": True, "checks": checks}, indent=2))
else:
    print(json.dumps({
        "use_gcs_direct": False,
        "message": "Skipping GCS auth because this session will use the Drive-local assets/raw_womd cache.",
    }, indent=2))


## Bootstrap Repository And Bind Artifacts

Clone or fast-forward `main`, validate the Drive mount, and bind the repository artifact paths to persistent Drive-backed storage.


In [ ]:
%%bash
set -euo pipefail

BOOTSTRAP=/tmp/latentdriver_colab_bootstrap.py
curl -fsSL \
  https://raw.githubusercontent.com/achyutmorang/latentdriver-waymax-experiments/main/scripts/colab_bootstrap.py \
  -o "$BOOTSTRAP"

python3 "$BOOTSTRAP" \
  --repo-url https://github.com/achyutmorang/latentdriver-waymax-experiments.git \
  --branch main \
  --repo-dir /content/latentdriver-waymax-experiments \
  --drive-base-root /content/drive/MyDrive/waymax_research \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


## List Available Runner Profiles

Print the supported `colab_canary.py` profiles. Use this when you need to confirm the exact profile names before launching a run.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
python3 scripts/colab_canary.py --list-profiles


## Confirm Git Revision

Fast-forward the local Colab checkout and print the active commit. This is a lightweight sanity check before expensive runs.


In [ ]:
%%bash
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main
git rev-parse --short HEAD


## Full Evaluation Dry-Run Gate

Run the fast readiness gate for `full_reactive` without launching simulation. This should report `ready: true` before the real full evaluation cell is used.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

# Optional readiness check. This should stay fast and should report ready: true before real full evaluation.
python3 scripts/colab_canary.py \
  --profile full-eval-dry-run \
  --model latentdriver_t2_j3 \
  --seed 0 \
  --auto-install-runtime \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


## Check Full Preprocess Markers

Verify that full preprocessing wrote `_SUCCESS` and `preprocess_manifest.json`. These markers prevent accidental full eval runs on partial preprocessing caches.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

PREPROCESS_DIR=/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path
test -f "$PREPROCESS_DIR/_SUCCESS" && echo "_SUCCESS present" || echo "_SUCCESS missing"
test -f "$PREPROCESS_DIR/preprocess_manifest.json" && echo "manifest present" || echo "manifest missing"


## Run First Full Reactive Simulation

Launch the next milestone: one resumable full reactive simulation/evaluation run for `latentdriver_t2_j3`. If Colab disconnects, rerun this same cell to skip completed shards and continue.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

# Next milestone: one resumable full reactive simulation/evaluation run for LatentDriver T2-J3.
# If Colab disconnects, rerun this same cell; completed shards are skipped.
python3 scripts/colab_canary.py \
  --profile full-eval-reactive-single \
  --model latentdriver_t2_j3 \
  --seed 0 \
  --auto-install-runtime \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


## Inspect Latest Debug Bundle

Use this after a failed canary run. It prints the latest run manifest and latest failure summary without requiring manual traceback hunting.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

# Use this after any failed canary run.
python3 - <<'PY'
import json
from pathlib import Path

latest = Path("results/debug_runs/latest/manifest.json")
latest_failure = Path("results/debug_runs/latest_failure/failure_summary.json")
for path in (latest, latest_failure):
    print(f"\n== {path} ==")
    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2)[:12000])
    else:
        print("missing")
PY
